# 05 Angle Estimation
DBF 与 MUSIC 空间谱对比可视化。

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.processing.angle_estimation import dbf_spectrum, music_spectrum
from src.visualization.spectrum_plots import plot_music_spectrum

num_ant = 8
true_angle = 25.0  # degrees
angles = np.linspace(-60, 60, 361)

# Single snapshot at the true angle
snapshot = np.exp(1j * 2 * np.pi * 0.5 * np.arange(num_ant) * np.sin(np.deg2rad(true_angle)))

# DBF spectrum
p_dbf = dbf_spectrum(snapshot, angles)
p_dbf_norm = p_dbf / np.max(p_dbf)

# MUSIC spectrum (multi-snapshot for better covariance estimate)
noise_std = 0.05
snapshots = snapshot[:, None] + noise_std * (
    np.random.randn(num_ant, 64) + 1j * np.random.randn(num_ant, 64)
)
p_music = music_spectrum(snapshots, angles, num_sources=1)

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, spectrum, label, color in zip(
        axes,
        [p_dbf_norm, p_music],
        ['DBF', 'MUSIC'],
        ['steelblue', 'darkorange']):
    ax.plot(angles, spectrum, lw=1.5, color=color)
    ax.axvline(true_angle, color='red', lw=1.0, linestyle='--', label=f'True angle: {true_angle}°')
    ax.set_xlabel('Angle (deg)')
    ax.set_ylabel('Normalized Spectrum')
    ax.set_title(f'{label} Spectrum')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Spatial Spectrum – DBF vs MUSIC', fontsize=13)
fig.tight_layout()
plt.show()

print(f'DBF  peak angle: {angles[np.argmax(p_dbf)]:.1f}°')
print(f'MUSIC peak angle: {angles[np.argmax(p_music)]:.1f}°')